# Attention-Based Image Captioning on Flickr8k — PyTorch Notebook

This notebook builds a Show, Attend and Tell-style captioner with a **frozen ImageNet CNN**, additive (Bahdanau) attention, an LSTMCell decoder, teacher forcing, doubly-stochastic attention regularization, greedy decoding, beam search, BLEU/METEOR evaluation, and attention-map visualization.

## 0. Imports

In [ ]:
import os, re, json, math, random, string, time, copy
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional, Sequence, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import models, transforms, io

from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

## 1. Hyperparams

In [ ]:
BATCH_SIZE = 32 if torch.cuda.is_available() else 4
NUM_WORKERS = 0
MAX_LEN = 32
MIN_FREQ = 2

PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGE_SIZE = 224
MAX_EPOCHS = 2
LEARNING_RATE = 1e-4

DATA_ROOT = Path("flickr8k")
IMAGE_DIR = DATA_ROOT / "Images"
CAPTION_FILE = DATA_ROOT / "captions.txt"
CHECKPOINT_PATH = DATA_ROOT / "best_attention_captioner.pt"


EMBED_DIM = 256
HIDDEN_DIM = 512
ATTENTION_DIM = 256
LAMBDA_DS = 1.0
EARLY_STOPPING_PATIENCE = 5

## 2. Data extraction and preprocessing

In [ ]:
def normalize_caption(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9']+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def load_flickr8k_captions(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.lower().startswith("image,caption"):
                continue
            m = re.match(r"^(.*?),\d+\s+(.*)$", line)
            if m:
                image, caption = m.group(1).strip(), m.group(2).strip()
            else:
                parts = line.split(",", 1)
                if len(parts) != 2:
                    continue
                image, caption = parts[0].strip(), parts[1].strip()
            rows.append((image, normalize_caption(caption)))
    return pd.DataFrame(rows, columns=["image", "caption"])

class Vocabulary:
    def __init__(self, min_freq=2):
        self.itos = ["<pad>", "<bos>", "<eos>", "<unk>"]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.min_freq = min_freq

    def build(self, texts):
        counts = Counter()
        for text in texts:
            counts.update(text.split())
        for token, freq in sorted(counts.items()):
            if freq >= self.min_freq and token not in self.stoi:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)

    def encode(self, text, max_len=MAX_LEN):
        words = text.split()[:max_len - 2]
        return [BOS_ID] + [self.stoi.get(w, UNK_ID) for w in words] + [EOS_ID]

    def decode(self, ids):
        words = []
        for x in ids:
            x = int(x)
            if x in (PAD_ID, BOS_ID):
                continue
            if x == EOS_ID:
                break
            if 0 <= x < len(self.itos):
                words.append(self.itos[x])
        return " ".join(words)

    def __len__(self):
        return len(self.itos)

In [ ]:
captions_df = load_flickr8k_captions(CAPTION_FILE)
images = captions_df["image"].unique().tolist()

rng = np.random.default_rng(42)
rng.shuffle(images)

n = len(images)
n_train = int(0.80 * n)
n_val = int(0.10 * n)

train_images = set(images[:n_train])
val_images = set(images[n_train:n_train + n_val])
test_images = set(images[n_train + n_val:])

train_df = captions_df[captions_df.image.isin(train_images)].reset_index(drop=True)
val_df = captions_df[captions_df.image.isin(val_images)].reset_index(drop=True)
test_df = captions_df[captions_df.image.isin(test_images)].reset_index(drop=True)

vocab = Vocabulary(MIN_FREQ)
vocab.build(train_df.caption)
VOCAB_SIZE = len(vocab)

print("Caption rows:", len(captions_df))
print("Images:", captions_df.image.nunique())
print("Vocab size:", VOCAB_SIZE)
print("Splits:", len(train_images), len(val_images), len(test_images))

## 3. Saving features and loading from there

### Encoder

In [ ]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self, backbone="vgg16", pretrained=True, output_size=(14, 14)):
        super().__init__()
        backbone = backbone.lower()
        self.backbone_name = backbone
        self.output_size = output_size

        if backbone == "vgg16":
            weights = models.VGG16_Weights.DEFAULT if pretrained else None
            base = models.vgg16(weights=weights)
            self.features = nn.Sequential(*list(base.features.children())[:30])
            self.encoder_dim = 512
        elif backbone == "vgg19":
            weights = models.VGG19_Weights.DEFAULT if pretrained else None
            base = models.vgg19(weights=weights)
            self.features = nn.Sequential(*list(base.features.children())[:36])
            self.encoder_dim = 512
        elif backbone == "resnet50":
            weights = models.ResNet50_Weights.DEFAULT if pretrained else None
            base = models.resnet50(weights=weights)
            self.features = nn.Sequential(*list(base.children())[:-2])
            self.encoder_dim = 2048
        elif backbone == "inception_v3":
            weights = models.Inception_V3_Weights.DEFAULT if pretrained else None
            base = models.inception_v3(weights=weights, aux_logits=False)
            # Keep the convolutional trunk; remove global average pool, dropout and fc.
            self.features = nn.Sequential(
                base.Conv2d_1a_3x3, base.Conv2d_2a_3x3, base.Conv2d_2b_3x3,
                nn.MaxPool2d(kernel_size=3, stride=2), base.Conv2d_3b_1x1,
                base.Conv2d_4a_3x3, nn.MaxPool2d(kernel_size=3, stride=2),
                base.Mixed_5b, base.Mixed_5c, base.Mixed_5d,
                base.Mixed_6a, base.Mixed_6b, base.Mixed_6c, base.Mixed_6d, base.Mixed_6e,
                base.Mixed_7a, base.Mixed_7b, base.Mixed_7c,
            )
            self.encoder_dim = 2048
        else:
            raise ValueError("backbone must be vgg16, vgg19, resnet50, or inception_v3")

        self.adaptive_pool = nn.AdaptiveAvgPool2d(output_size) if output_size is not None else nn.Identity()
        for param in self.parameters():
            param.requires_grad = False
        self.eval()

    def train(self, mode=True):
        # Frozen pretrained backbone stays in eval mode (e.g. BatchNorm behaviour stays fixed).
        super().train(False)
        return self

    @torch.no_grad()
    def forward(self, images):
        feature_map = self.features(images)
        feature_map = self.adaptive_pool(feature_map)
        # (B, D, H, W) -> (B, H*W, D), i.e. annotation vectors a_i
        return feature_map.flatten(2).transpose(1, 2).contiguous()

BACKBONE = "vgg16"
ADAPTIVE_OUTPUT_SIZE = (14, 14)  # use None for native ResNet50 7x7x2048 -> (49, 2048)
encoder = None
encoder = CNNFeatureExtractor(BACKBONE, pretrained=True, output_size=ADAPTIVE_OUTPUT_SIZE).to(DEVICE)
encoder.eval()
n_trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in encoder.parameters())
print(encoder)
print(f"Encoder parameters: {n_total:,} total; {n_trainable:,} trainable")

### Feature extraction

In [ ]:
from tqdm import tqdm
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
class MyImageDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = IMAGE_DIR / row.image
        image = io.read_image(str(path), mode=io.ImageReadMode.RGB).float() / 255.0
        image = image_transform(image)
        return idx, image

def save_features(df, encoder, name="train"):
    loader = DataLoader(MyImageDataset(df), batch_size=BATCH_SIZE, num_workers=0, shuffle=False)
    num_rows = len(loader.dataset)
    num_regions = ADAPTIVE_OUTPUT_SIZE[0] * ADAPTIVE_OUTPUT_SIZE[1]   # 196
    data = np.lib.format.open_memmap(
        f"{name}_feat_data.npy", 
        mode='w+', 
        dtype='float32', 
        shape=(num_rows, num_regions, encoder.encoder_dim)
    )
    encoder = encoder.to(DEVICE).eval()
    for idx, img in tqdm(loader, desc=f"{name} feature processing"):
        img = img.to(DEVICE)
        feat = encoder(img)
        data[idx.numpy()] = feat.cpu().numpy()
        data.flush()
    del data
    print("Finished")

# save_features(train_df, encoder, "train")
save_features(val_df, encoder, "val")
save_features(test_df, encoder, "test")

### Dataloaders containing feature and captions

In [ ]:
del encoder

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class FlickrFeatureCaptionDataset(Dataset):
    def __init__(self, frame, data_path="train_feat_data.npy"):
        self.frame = frame.reset_index(drop=True)
        self.data = np.load(data_path, mmap_mode="r")
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        feat = torch.from_numpy(np.array(self.data[idx]))   # copy out of memmap -> tensor
        sequence = vocab.encode(row.caption)
        caption = torch.tensor(sequence, dtype=torch.long)
        return {"feature": feat, "caption": caption, "length": len(sequence)}

def caption_collate_fn(batch):
    batch = sorted(batch, key=lambda item: item["length"], reverse=True)
    features = torch.stack([item["feature"] for item in batch], dim=0)   # (B, L, D)
    captions = [item["caption"] for item in batch]
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    captions = pad_sequence(captions, batch_first=True, padding_value=PAD_ID)
    return features, captions, lengths

train_loader = DataLoader(FlickrFeatureCaptionDataset(train_df, "train_feat_data.npy"), batch_size=BATCH_SIZE, shuffle=True, collate_fn=caption_collate_fn)
val_loader = DataLoader(FlickrFeatureCaptionDataset(val_df, "val_feat_data.npy"), batch_size=BATCH_SIZE, shuffle=False, collate_fn=caption_collate_fn)
test_loader = DataLoader(FlickrFeatureCaptionDataset(test_df, "test_feat_data.npy"), batch_size=BATCH_SIZE, shuffle=False, collate_fn=caption_collate_fn)

print(len(train_loader), len(val_loader), len(test_loader))
print(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

# Part III — Attention and decoder

## 9. Soft Bahdanau attention, hard attention, and beta gate

Given annotation vectors `a` with shape `(B,L,D)` and the previous hidden state `h` with shape `(B,H)`, additive attention computes a scalar score for every location, normalizes scores over locations, and returns a weighted context vector. With hard attention, a location is sampled from `Categorical(probs=alpha)`; the sampled location's log-probability is retained for REINFORCE.

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, encoder_dim, hidden_dim, attention_dim=256, use_beta=True):
        super().__init__()
        self.encoder_proj = nn.Linear(encoder_dim, attention_dim, bias=False)
        self.hidden_proj = nn.Linear(hidden_dim, attention_dim, bias=False)
        self.score_proj = nn.Linear(attention_dim, 1, bias=False)
        self.use_beta = use_beta
        self.beta_proj = nn.Linear(hidden_dim, 1) if use_beta else None

    def forward(self, annotations, previous_hidden, hard=False):
        # annotations: (B,L,D); previous_hidden: (B,H)
        energy = self.score_proj(torch.tanh(
            self.encoder_proj(annotations) + self.hidden_proj(previous_hidden).unsqueeze(1)
        )).squeeze(-1)  # (B,L)
        alpha = torch.softmax(energy, dim=1)
        distribution = torch.distributions.Categorical(probs=alpha)
        entropy = distribution.entropy()  # (B,)

        if hard:
            selected_location = distribution.sample()  # (B,)
            batch_idx = torch.arange(annotations.size(0), device=annotations.device)
            context = annotations[batch_idx, selected_location]  # (B,D)
            log_prob = distribution.log_prob(selected_location)  # (B,)
        else:
            context = torch.sum(alpha.unsqueeze(-1) * annotations, dim=1)  # (B,D)
            log_prob = None

        beta = torch.sigmoid(self.beta_proj(previous_hidden)) if self.use_beta else context.new_ones((context.size(0), 1))
        context = beta * context
        return context, alpha, beta.squeeze(-1), log_prob, entropy


# Shape / probability sanity check.
attn_demo = BahdanauAttention(encoder_dim=512, hidden_dim=256, attention_dim=128).to(DEVICE)
a_demo = torch.randn(4, 196, 512, device=DEVICE)
h_demo = torch.randn(4, 256, device=DEVICE)
context_demo, alpha_demo, beta_demo, _, entropy_demo = attn_demo(a_demo, h_demo)
print("Context shape:", tuple(context_demo.shape))
print("Attention shape at one time step:", tuple(alpha_demo.shape))  # (B,L) = (4,196)
print("alpha.sum(dim=1):", alpha_demo.sum(dim=1))
print("All sums approximately one:", torch.allclose(alpha_demo.sum(dim=1), torch.ones(4, device=DEVICE), atol=1e-6))
print("Beta shape:", tuple(beta_demo.shape))

### Doubly-stochastic attention penalty

For a sequence of attention maps `alpha` with shape `(B,T,L)`, the penalty is

$$\lambda\,\frac{1}{BL}\sum_{b,i}\left(1-\sum_t \alpha_{b,t,i}\right)^2$$

The time sum asks the decoder to visit spatial locations across the generated sentence. This is different from the per-timestep normalization check `alpha_t.sum(dim=1) == 1`, which sums over **locations** and is already enforced by softmax. Padding time steps must be masked out.

In [ ]:
def doubly_stochastic_penalty(attention_weights, lengths=None, lambda_ds=1.0):
    """attention_weights: (B,T,L), lengths are caption lengths including <start>/<end>."""
    if lambda_ds == 0:
        return attention_weights.new_zeros(())
    alpha = attention_weights
    if lengths is not None:
        steps = alpha.size(1)
        valid_steps = torch.arange(steps, device=alpha.device).unsqueeze(0) < (lengths - 1).unsqueeze(1)
        alpha = alpha * valid_steps.unsqueeze(-1).to(alpha.dtype)
    # Sum over T for each location L, not over L at an individual timestep.
    location_coverage = alpha.sum(dim=1)  # (B,L)
    return lambda_ds * ((1.0 - location_coverage) ** 2).mean()

alpha_sequence_demo = torch.softmax(torch.randn(3, 7, 196, device=DEVICE), dim=-1)
lengths_demo = torch.tensor([8, 6, 5], device=DEVICE)
print("Doubly-stochastic penalty:", float(doubly_stochastic_penalty(alpha_sequence_demo, lengths_demo)))

## 10. Attention decoder with `nn.Embedding` and `nn.LSTMCell`

The caption tensor includes `<start>` and `<end>`. During training, input tokens are `captions[:, :-1]` and targets are `captions[:, 1:]`. At time `t`, the decoder uses the previous hidden state to attend, concatenates `[embedding(y_{t-1}), context_t]`, updates the recurrent cell, and computes a deep output:

$$\mathrm{logits}_t = L_o\left(\tanh(Ey_{t-1}+L_hh_t+L_z\hat z_t)\right)$$

Only the still-active prefix of each length-sorted batch is decoded at later steps. `cell_type="gru"`, `use_attention=False`, and `use_beta=False` are included as ablation switches.

In [ ]:
class AttentionCaptioningDecoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        encoder_dim=512,
        embed_dim=256,
        hidden_dim=512,
        attention_dim=256,
        dropout=0.5,
        pad_idx=PAD_ID,
        use_beta=True,
        use_attention=True,
        cell_type="lstm",
    ):
        super().__init__()
        if cell_type not in {"lstm", "gru"}:
            raise ValueError("cell_type must be 'lstm' or 'gru'")
        self.vocab_size = vocab_size
        self.encoder_dim = encoder_dim
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.pad_idx = pad_idx
        self.use_attention = use_attention
        self.cell_type = cell_type

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.init_h = nn.Linear(encoder_dim, hidden_dim)
        self.init_c = nn.Linear(encoder_dim, hidden_dim) if cell_type == "lstm" else None
        self.attention = BahdanauAttention(encoder_dim, hidden_dim, attention_dim, use_beta) if use_attention else None
        cell_cls = nn.LSTMCell if cell_type == "lstm" else nn.GRUCell
        self.rnn_cell = cell_cls(embed_dim + encoder_dim, hidden_dim)

        # Deep output layer: Lo(tanh(E*y + Lh*h + Lz*z)), with dropout before Lo.
        self.embed_proj = nn.Linear(embed_dim, embed_dim)
        self.hidden_proj = nn.Linear(hidden_dim, embed_dim)
        self.context_proj = nn.Linear(encoder_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(embed_dim, vocab_size)

    def init_states(self, annotations):
        mean_annotations = annotations.mean(dim=1)
        h = torch.tanh(self.init_h(mean_annotations))
        if self.cell_type == "lstm":
            c = torch.tanh(self.init_c(mean_annotations))
        else:
            c = None
        return h, c

    def step(self, annotations, previous_token, h, c=None, hard_attention=False):
        """One autoregressive step. annotations (B,L,D), previous_token (B,)."""
        if self.use_attention:
            context, alpha, beta, log_prob, entropy = self.attention(annotations, h, hard=hard_attention)
        else:
            context = annotations.mean(dim=1)
            alpha = annotations.new_full((annotations.size(0), annotations.size(1)), 1.0 / annotations.size(1))
            beta = annotations.new_ones((annotations.size(0),))
            log_prob = None
            entropy = annotations.new_zeros((annotations.size(0),))

        embedding = self.embedding(previous_token)
        rnn_input = torch.cat([embedding, context], dim=1)
        if self.cell_type == "lstm":
            h_new, c_new = self.rnn_cell(rnn_input, (h, c))
        else:
            h_new = self.rnn_cell(rnn_input, h)
            c_new = None

        deep_output = torch.tanh(
            self.embed_proj(embedding) + self.hidden_proj(h_new) + self.context_proj(context)
        )
        logits = self.output_proj(self.dropout(deep_output))
        return logits, h_new, c_new, alpha, beta, log_prob, entropy

    def forward(self, annotations, captions, lengths, hard_attention=False):
        """Teacher-forced forward; returns logits (B,T,V), alphas (B,T,L), logp, entropy."""
        batch_size, n_locations, _ = annotations.shape
        max_steps = max(0, captions.size(1) - 1)
        h, c = self.init_states(annotations)
        logits_all = annotations.new_zeros((batch_size, max_steps, self.vocab_size))
        alpha_all = annotations.new_zeros((batch_size, max_steps, n_locations))
        logp_all = annotations.new_zeros((batch_size, max_steps))
        entropy_all = annotations.new_zeros((batch_size, max_steps))

        for t in range(max_steps):
            # A target exists at position t+1 exactly when caption_length > t+1.
            batch_size_t = int((lengths > (t + 1)).sum().item())
            if batch_size_t == 0:
                break
            active_annotations = annotations[:batch_size_t]
            h_active = h[:batch_size_t]
            c_active = c[:batch_size_t] if c is not None else None
            prev_tokens = captions[:batch_size_t, t]
            (step_logits, h_new, c_new, alpha, _beta, logp, entropy) = self.step(
                active_annotations, prev_tokens, h_active, c_active, hard_attention=hard_attention
            )
            logits_all[:batch_size_t, t] = step_logits
            alpha_all[:batch_size_t, t] = alpha
            if logp is not None:
                logp_all[:batch_size_t, t] = logp
            entropy_all[:batch_size_t, t] = entropy
            # Active examples remain a sorted prefix, so state tensors may shrink each step.
            h = h_new
            c = c_new

        return logits_all, alpha_all, logp_all, entropy_all


# Infer encoder dimensions from the selected configuration or cache.
_feat = next(iter(train_loader))[0][0]
ENCODER_DIM = _feat.shape[1]
NUM_LOCATIONS = _feat.shape[0]


model = AttentionCaptioningDecoder(
    vocab_size=VOCAB_SIZE, encoder_dim=ENCODER_DIM, embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM, attention_dim=ATTENTION_DIM, dropout=0.5,
    pad_idx=PAD_ID, use_beta=True, use_attention=True, cell_type="lstm",
).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Decoder parameters: {total_params:,} total; {trainable_params:,} trainable")

dummy_features = torch.randn(2, NUM_LOCATIONS, ENCODER_DIM, device=DEVICE)
dummy_caps = torch.tensor([[BOS_ID, 10, EOS_ID], [BOS_ID, EOS_ID, PAD_ID]], device=DEVICE)
dummy_lengths = torch.tensor([3, 2], device=DEVICE)
model.eval()
with torch.no_grad():
    dummy_logits, dummy_alpha, _, _ = model(dummy_features, dummy_caps, dummy_lengths)
print("Dummy logits:", tuple(dummy_logits.shape), "attention:", tuple(dummy_alpha.shape))

# Optional package-based summary (fallback above works without torchinfo).
try:
    from torchinfo import summary as torchinfo_summary
    torchinfo_summary(model, input_data=(dummy_features, dummy_caps, dummy_lengths), verbose=0)
except ImportError:
    print("Optional torchinfo not installed; printed module tree and parameter count instead.")

### Optional GloVe initialisation

Use a GloVe file whose vector dimension equals `EMBED_DIM`. If using 300-dimensional GloVe, construct the decoder with `embed_dim=300` before loading vectors. Unmatched words keep their random initialisation.

In [ ]:
def initialise_embedding_from_glove(embedding, vocab, glove_name=None):
    import gensim.downloader as gensim_api
    dim = embedding.embedding_dim
    if glove_name is None:
        if dim not in (50, 100, 200, 300):
            raise ValueError(f"No GloVe model with dim {dim}. Use 50, 100, 200 or 300.")
        glove_name = f"glove-wiki-gigaword-{dim}"
    glove = gensim_api.load(glove_name)
    copied = 0
    with torch.no_grad():
        for word, idx in vocab.items():
            # GloVe wiki-gigaword is lowercase
            key = word.lower()
            if key in glove.key_to_index:
                vector = torch.from_numpy(np.asarray(glove[key], dtype=np.float32))
                embedding.weight[idx].copy_(vector.to(embedding.weight.dtype))
                copied += 1
    print(f"Initialised {copied:,}/{len(vocab):,} tokens from GloVe")
    return copied

# Usage
initialise_embedding_from_glove(model.embedding, vocab.stoi)

# Part IV — Training

## 11. Cross-entropy, doubly-stochastic penalty, top-5 accuracy, and REINFORCE helper

`CrossEntropyLoss(ignore_index=PAD_IDX)` masks padded targets. The attention penalty masks inactive caption steps. The hard-attention option adds a score-function policy-gradient term and an entropy bonus. In this implementation, the reward is negative per-sequence NLL to make the training mechanics demonstrable; metric-reward fine-tuning would instead use a sequence-level reward such as CIDEr/BLEU.

The requested update `b_k = 0.9 b_{k-1} + 0.1 log p` is exposed below as a literal helper. Note that a conventional REINFORCE baseline tracks a reward (and should not depend on the sampled action); using the sampled action's `log p` as the baseline is not the standard unbiased-baseline choice. The training loop therefore uses a moving average of reward by default.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)


def topk_token_accuracy(logits, targets, pad_idx=PAD_ID, k=5):
    """Token accuracy over non-pad targets, returning a fraction in [0,1]."""
    vocab_size = logits.size(-1)
    k = min(k, vocab_size)
    valid = targets.ne(pad_idx)
    if valid.sum() == 0:
        return 0.0
    predictions = logits.topk(k, dim=-1).indices
    correct = predictions.eq(targets.unsqueeze(-1)).any(dim=-1) & valid
    return (correct.sum().float() / valid.sum()).item()


def caption_loss_parts(logits, captions, lengths, alpha, lambda_ds=1.0, pad_idx=PAD_ID):
    targets = captions[:, 1:]
    logits = logits[:, :targets.size(1), :]
    ce_loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=pad_idx
    )
    ds_loss = doubly_stochastic_penalty(alpha[:, :targets.size(1)], lengths, lambda_ds=lambda_ds)
    top5 = topk_token_accuracy(logits.detach(), targets, pad_idx=pad_idx, k=5)
    return ce_loss, ds_loss, top5, targets


def moving_average_logprob_baseline(previous_baseline, mean_log_prob, momentum=0.9):
    """Literal helper for the requested b_k = 0.9*b_(k-1) + 0.1*mean(log p) update."""
    if previous_baseline is None:
        return float(mean_log_prob)
    return momentum * float(previous_baseline) + (1.0 - momentum) * float(mean_log_prob)


def train_one_epoch(model, loader, optimizer, device, lambda_ds=1.0, max_grad_norm=5.0,
                    hard_attention=False, entropy_weight=0.01, reward_baseline=None,
                    reinforce_baseline_mode="reward"):
    model.train()
    total_loss = total_ce = total_ds = total_policy = total_entropy = 0.0
    total_top5 = 0.0
    n_batches = 0
    baseline = reward_baseline

    for features, captions, lengths in tqdm(loader, desc="training"):
        features = features.to(device, non_blocking=True)
        captions = captions.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        logits, alpha, sampled_log_probs, entropies = model(
            features, captions, lengths, hard_attention=hard_attention
        )
        ce_loss, ds_loss, top5, targets = caption_loss_parts(
            logits, captions, lengths, alpha, lambda_ds=lambda_ds
        )
        policy_loss = logits.new_zeros(())
        entropy_bonus = logits.new_zeros(())

        if hard_attention and model.use_attention:
            # Token NLL per sequence; padding is excluded.
            token_loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                ignore_index=PAD_ID, reduction="none"
            ).view_as(targets)
            valid = targets.ne(PAD_ID)
            seq_nll = (token_loss * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)
            rewards = -seq_nll.detach()  # higher (less negative) is better
            baseline_tensor = logits.new_tensor(0.0 if baseline is None else baseline)
            advantages = rewards - baseline_tensor
            valid_steps = (torch.arange(sampled_log_probs.size(1), device=device).unsqueeze(0)
                           < (lengths - 1).unsqueeze(1)).to(sampled_log_probs.dtype)
            seq_log_probs = (sampled_log_probs * valid_steps).sum(dim=1)
            policy_loss = -(advantages * seq_log_probs).mean()
            entropy_per_sequence = (entropies * valid_steps).sum(dim=1) / valid_steps.sum(dim=1).clamp_min(1)
            entropy_bonus = -entropy_weight * entropy_per_sequence.mean()
            if reinforce_baseline_mode == "reward":
                batch_statistic = rewards.mean().item()
            elif reinforce_baseline_mode == "log_prob":
                # Literal requested variant: b_k = 0.9*b_(k-1) + 0.1*mean(log p).
                # This is not the standard unbiased REINFORCE baseline; use "reward" by default.
                denominator = valid_steps.sum().clamp_min(1.0)
                batch_statistic = ((sampled_log_probs.detach() * valid_steps).sum() / denominator).item()
            else:
                raise ValueError("reinforce_baseline_mode must be 'reward' or 'log_prob'")
            baseline = batch_statistic if baseline is None else 0.9 * baseline + 0.1 * batch_statistic

        loss = ce_loss + ds_loss + policy_loss + entropy_bonus
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()

        total_loss += loss.item()
        total_ce += ce_loss.item()
        total_ds += ds_loss.item()
        total_policy += policy_loss.item()
        total_entropy += entropy_bonus.item()
        total_top5 += top5
        n_batches += 1

    denom = max(1, n_batches)
    return {
        "loss": total_loss / denom,
        "ce": total_ce / denom,
        "ds": total_ds / denom,
        "policy": total_policy / denom,
        "entropy_term": total_entropy / denom,
        "top5": total_top5 / denom,
        "reward_baseline": baseline,
    }


@torch.no_grad()
def evaluate_loss(model, loader, device, lambda_ds=1.0):
    model.eval()
    total_loss = total_ce = total_ds = total_top5 = 0.0
    n_batches = 0
    for features, captions, lengths in tqdm(loader, desc="evaluating loss"):
        features = features.to(device, non_blocking=True)
        captions = captions.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)
        logits, alpha, _, _ = model(features, captions, lengths, hard_attention=False)
        ce_loss, ds_loss, top5, _ = caption_loss_parts(logits, captions, lengths, alpha, lambda_ds=lambda_ds)
        total_loss += (ce_loss + ds_loss).item()
        total_ce += ce_loss.item()
        total_ds += ds_loss.item()
        total_top5 += top5
        n_batches += 1
    denom = max(1, n_batches)
    return {"loss": total_loss / denom, "ce": total_ce / denom,
            "ds": total_ds / denom, "top5": total_top5 / denom}

## 12. Greedy decoding, beam search, and corpus BLEU helpers

These functions operate directly on pre-extracted `(L,D)` features. They are used for validation BLEU, test generation, and new-image captioning.

In [ ]:
@torch.no_grad()
def greedy_decode_features(model, features, max_len=MAX_LEN):
    """Return generated token IDs (including <start>, usually <end>) and per-step attention maps.
    ids, attn = greedy_decode_features(model, feat)
    caption = vocab.decode(ids)   # skips PAD/BOS, stops at EOS
    """
    model.eval()
    device = next(model.parameters()).device
    if features.dim() == 2:
        features = features.unsqueeze(0)
    features = features.to(device)
    h, c = model.init_states(features)
    current = torch.tensor([BOS_ID], dtype=torch.long, device=device)
    sequence = [BOS_ID]
    attention_maps = []
    for _ in range(max(1, max_len - 1)):
        logits, h, c, alpha, _beta, _log_prob, _entropy = model.step(
            features, current, h, c, hard_attention=False
        )
        next_token = int(logits.argmax(dim=-1).item())
        attention_maps.append(alpha[0].detach().cpu().numpy())
        sequence.append(next_token)
        if next_token == EOS_ID:
            break
        current = torch.tensor([next_token], dtype=torch.long, device=device)
    return sequence, attention_maps


def _length_penalized_score(log_score, sequence, alpha=0.7):
    generated_length = max(1, len(sequence))
    penalty = ((5.0 + generated_length) / 6.0) ** alpha
    return log_score / penalty


@torch.no_grad()
def beam_search_decode_features(model, features, vocab, beam_size=3,
                                max_len=MAX_LEN, length_penalty=0.7, device=DEVICE):
    """Simple single-image beam search. Returns best token IDs and aligned attention maps."""
    model.eval()
    if features.dim() == 2:
        features = features.unsqueeze(0)
    features = features.to(device)
    h0, c0 = model.init_states(features)
    # Each beam: (sequence, cumulative_logprob, hidden, cell, attention_maps)
    beams = [([BOS_ID], 0.0, h0, c0, [])]
    for _ in range(max(1, max_len - 1)):
        candidates = []
        for sequence, score, h, c, maps in beams:
            if sequence[-1] == EOS_ID:
                # finished: carry forward unchanged so it competes with the rest
                candidates.append((sequence, score, h, c, maps))
                continue
            prev = torch.tensor([sequence[-1]], dtype=torch.long, device=device)
            logits, h_new, c_new, alpha, _beta, _log_prob, _entropy = model.step(
                features, prev, h, c, hard_attention=False
            )
            log_probs = F.log_softmax(logits, dim=-1)
            top_scores, top_tokens = torch.topk(log_probs[0], k=min(beam_size, log_probs.size(-1)))
            attn = alpha[0].detach().cpu().numpy()
            for token_score, token_id in zip(top_scores.tolist(), top_tokens.tolist()):
                candidates.append((
                    sequence + [int(token_id)],
                    score + float(token_score),
                    h_new, c_new,
                    maps + [attn],
                ))
        candidates.sort(
            key=lambda item: _length_penalized_score(item[1], item[0], length_penalty),
            reverse=True,
        )
        beams = candidates[:beam_size]
        if all(item[0][-1] == EOS_ID for item in beams):
            break
    best = max(beams, key=lambda item: _length_penalized_score(item[1], item[0], length_penalty))
    return best[0], best[4]


def corpus_bleu_score(references, hypotheses, order=4):
    weights_by_order = {
        1: (1.0, 0.0, 0.0, 0.0),
        2: (0.5, 0.5, 0.0, 0.0),
        3: (1/3, 1/3, 1/3, 0.0),
        4: (0.25, 0.25, 0.25, 0.25),
    }
    return corpus_bleu(references, hypotheses, weights=weights_by_order[order],
                       smoothing_function=SmoothingFunction().method1)


import hashlib
@torch.no_grad()
def evaluate_corpus_bleu(model, loader, vocab,
                         beam_size=1, max_len=MAX_LEN, device=DEVICE):
    model.eval()
    hyp_by_img = {}    # feature-hash -> hypothesis tokens
    refs_by_img = {}   # feature-hash -> list of reference token lists
    for features, caption_ids, lengths in tqdm(loader, desc="calculating bleu"):
        for i in range(features.size(0)):
            feat = features[i]                                   # (L, D)
            key = hashlib.md5(feat.numpy().tobytes()).hexdigest()
            # reference: this caption (BOS/EOS/PAD stripped by vocab.decode)
            ref = vocab.decode(caption_ids[i, :lengths[i]].tolist()).split()
            refs_by_img.setdefault(key, []).append(ref)
            # decode each unique image only once
            if key not in hyp_by_img:
                if beam_size <= 1:
                    seq, _ = greedy_decode_features(model, feat, max_len=max_len)
                else:
                    seq, _ = beam_search_decode_features(
                        model, feat, vocab, beam_size=beam_size,
                        max_len=max_len, device=device)
                hyp_by_img[key] = vocab.decode(seq).split()
    keys = list(hyp_by_img.keys())
    hypotheses = [hyp_by_img[k] for k in keys]
    references = [refs_by_img[k] for k in keys]
    scores = {f"BLEU-{n}": corpus_bleu_score(references, hypotheses, order=n)
              for n in range(1, 5)}
    return scores, references, hypotheses

## 13. Full training loop, scheduler, validation-BLEU early stopping, checkpointing

For Flickr8k the original Show, Attend and Tell work used RMSProp; Adam is also a reasonable option for a modern reproduction. Here the default is RMSProp with learning rate `4e-4`. The scheduler and early stopping monitor **validation BLEU-4**, not validation loss.

In [ ]:
def fit_captioner(
    model, train_loader, val_loader,
    vocab, epochs=MAX_EPOCHS, learning_rate=LEARNING_RATE,
    optimizer_name="RMSprop", lambda_ds=LAMBDA_DS, checkpoint_path=CHECKPOINT_PATH,
    patience=EARLY_STOPPING_PATIENCE, hard_attention=False, entropy_weight=0.01,
    reinforce_baseline_mode="reward",
):
    params = [p for p in model.parameters() if p.requires_grad]
    device = next(model.parameters()).device
    if optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(params, lr=learning_rate)
    elif optimizer_name.lower() == "rmsprop":
        optimizer = torch.optim.RMSprop(params, lr=learning_rate)
    else:
        raise ValueError("optimizer_name must be 'Adam' or 'RMSprop'")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )

    history = []
    best_bleu = -float("inf")
    best_epoch = 0
    no_improve = 0
    reward_baseline = None
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, device, lambda_ds=lambda_ds,
            max_grad_norm=5.0, hard_attention=hard_attention,
            entropy_weight=entropy_weight, reward_baseline=reward_baseline,
            reinforce_baseline_mode=reinforce_baseline_mode,
        )
        reward_baseline = train_metrics.get("reward_baseline")
        val_metrics = evaluate_loss(model, val_loader, device, lambda_ds=lambda_ds)
        val_bleu_scores, _, _ = evaluate_corpus_bleu(
            model, val_loader, vocab,
            beam_size=1, max_len=MAX_LEN, device=device,
        )
        val_bleu4 = val_bleu_scores["BLEU-4"]
        scheduler.step(val_bleu4)
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_ce": train_metrics["ce"],
            "train_ds": train_metrics["ds"],
            "train_top5": train_metrics["top5"],
            "val_loss": val_metrics["loss"],
            "val_top5": val_metrics["top5"],
            "val_bleu1": val_bleu_scores["BLEU-1"],
            "val_bleu2": val_bleu_scores["BLEU-2"],
            "val_bleu3": val_bleu_scores["BLEU-3"],
            "val_bleu4": val_bleu4,
            "lr": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - start_time,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{epochs} | train={row['train_loss']:.4f} "
            f"val={row['val_loss']:.4f} | val BLEU-4={val_bleu4:.4f} "
            f"| top-5={row['train_top5']:.3f} | lr={row['lr']:.2e} "
            f"| {row['seconds']:.1f}s"
        )

        if val_bleu4 > best_bleu:
            best_bleu = val_bleu4
            best_epoch = epoch
            no_improve = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_bleu": best_bleu,
                "word2idx": vocab.stoi,
                "idx2word": vocab.itos,
                "config": {
                    "encoder_dim": model.encoder_dim,
                    "embed_dim": model.embed_dim,
                    "hidden_dim": model.hidden_dim,
                    "use_attention": model.use_attention,
                    "cell_type": model.cell_type,
                },
            }, checkpoint_path)
            print(f"  Saved best checkpoint: {checkpoint_path}")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping: validation BLEU-4 did not improve for {patience} epochs.")
                break

    history_df = pd.DataFrame(history)
    print(f"Best validation BLEU-4={best_bleu:.4f} at epoch {best_epoch}")
    return history_df, optimizer


def load_captioner_checkpoint(model, checkpoint_path, optimizer=None, map_location=DEVICE):
    checkpoint = torch.load(checkpoint_path, map_location=map_location, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    print(f"Loaded epoch {checkpoint.get('epoch')} with best BLEU {checkpoint.get('best_bleu', float('nan')):.4f}")
    return checkpoint


history_df = None
optimizer = None
history_df, optimizer = fit_captioner(
    model, train_loader, val_loader, vocab, epochs=MAX_EPOCHS, learning_rate=LEARNING_RATE,
    optimizer_name="RMSprop", lambda_ds=LAMBDA_DS, checkpoint_path=CHECKPOINT_PATH,
)

In [ ]:
if history_df is not None and not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="validation")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training and validation loss")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["val_bleu4"], marker="o", label="validation BLEU-4")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("BLEU-4")
    axes[1].set_title("Validation BLEU-4 (early-stopping metric)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("Train the model to populate history_df and display learning curves.")

# Part V — Inference and caption generation

## 15. Generate captions for the test set and compare BLEU-1…4

The evaluation uses five tokenized references per image and one tokenized hypothesis per image, as required by `corpus_bleu`. The decoder has only one vocabulary-based output token stream; reference captions remain cleaned text tokens, excluding the special tokens.

In [ ]:
class ImageFeatureStore:
    """Image-level view over a row-aligned feature .npy (one row per caption in `frame`)."""
    def __init__(self, frame, data_path):
        self.frame = frame.reset_index(drop=True)
        self.data = np.load(data_path, mmap_mode="r")
        first = self.frame.drop_duplicates("image")
        self.id_to_row = dict(zip(first["image"], first.index))      # image -> first feature row
        self.captions = self.frame.groupby("image")["caption"].apply(list).to_dict()
        self.shape = self.data.shape

    def get(self, image_id):
        return torch.from_numpy(np.array(self.data[self.id_to_row[image_id]]))   # (L, D)


def image_path_for(image_id):
    return IMAGE_DIR / image_id

def references_as_tokens(image_id, caption_map):
    return [c.split() for c in caption_map[image_id]]


test_store = ImageFeatureStore(test_df, "test_feat_data.npy")
test_ids = list(test_store.id_to_row)
captions_clean = test_store.captions

In [ ]:
def score_caption_dicts(image_ids, hypotheses_by_id, clean_caption_map):
    references, hypotheses = [], []
    used_ids = []
    for image_id in image_ids:
        if image_id not in hypotheses_by_id or image_id not in clean_caption_map:
            continue
        used_ids.append(image_id)
        references.append(references_as_tokens(image_id, clean_caption_map))
        hypotheses.append(hypotheses_by_id[image_id])
    scores = {f"BLEU-{n}": corpus_bleu_score(references, hypotheses, n) for n in range(1, 5)}
    return scores, references, hypotheses, used_ids

@torch.no_grad()
def generate_captions_for_ids(model, image_ids, store, vocab,
                              method="greedy", beam_size=3, max_len=MAX_LEN, device=DEVICE):
    results, attention_results = {}, {}
    for image_id in tqdm(image_ids, desc="generating caption"):
        if image_id not in store.id_to_row:
            continue
        feature = store.get(image_id)
        if method == "greedy":
            seq, maps = greedy_decode_features(model, feature, max_len=max_len)
        elif method == "beam":
            seq, maps = beam_search_decode_features(
                model, feature, vocab, beam_size=beam_size, max_len=max_len, device=device)
        else:
            raise ValueError("method must be 'greedy' or 'beam'")
        results[image_id] = vocab.decode(seq).split()
        attention_results[image_id] = (seq, maps)
    return results, attention_results


test_greedy = test_beam = None
if CHECKPOINT_PATH.exists():
    load_captioner_checkpoint(model, CHECKPOINT_PATH)
    test_greedy, greedy_attention_results = generate_captions_for_ids(
        model, test_ids, test_store, vocab, method="greedy")
    test_beam, beam_attention_results = generate_captions_for_ids(
        model, test_ids, test_store, vocab, method="beam", beam_size=3)
    greedy_scores, greedy_refs, greedy_hyps, eval_ids = score_caption_dicts(test_ids, test_greedy, captions_clean)
    beam_scores, beam_refs, beam_hyps, _ = score_caption_dicts(test_ids, test_beam, captions_clean)
    results_table = pd.DataFrame([greedy_scores, beam_scores], index=["Greedy", "Beam-3"])
    display(results_table)
else:
    print("Test evaluation waits for a trained checkpoint.")

## 16. Sentence BLEU, METEOR, and optional external metrics

Corpus BLEU aggregates n-gram counts and brevity penalty across the corpus; `sentence_bleu` computes a score for one image and can be much noisier. NLTK's METEOR expects token sequences. The first METEOR call may require WordNet data.

In [ ]:
def evaluate_meteor(references, hypotheses):
    """Mean image-level METEOR; downloads WordNet if it is not already present."""
    import nltk
    try:
        nltk.data.find("corpora/wordnet")
    except LookupError:
        nltk.download("wordnet", quiet=True)
    scores = [meteor_score(refs, hyp) for refs, hyp in zip(references, hypotheses)]
    return float(np.mean(scores)) if scores else float("nan")


if 'greedy_refs' in globals() and 'greedy_hyps' in globals() and greedy_refs:
    print("Greedy corpus BLEU-1..4:", {n: corpus_bleu_score(greedy_refs, greedy_hyps, n) for n in range(1, 5)})
    print("Greedy mean METEOR:", evaluate_meteor(greedy_refs, greedy_hyps))
    idx = 0
    one_sentence_bleu4 = sentence_bleu(
        greedy_refs[idx], greedy_hyps[idx], weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=SmoothingFunction().method1,
    )
    print("First image sentence BLEU-4:", one_sentence_bleu4)
    print("Corpus BLEU-4:", corpus_bleu_score(greedy_refs, greedy_hyps, 4))
else:
    print("Run test-set generation first to calculate test metrics.")

# Optional: torchmetrics BLEU (API varies by version; use only if installed).
try:
    from torchmetrics.text import BLEUScore
    if 'greedy_hyps' in globals() and greedy_hyps:
        metric = BLEUScore(n_gram=4)
        preds = [" ".join(x) for x in greedy_hyps]
        target_lists = [[" ".join(ref) for ref in refs] for refs in greedy_refs]
        print("torchmetrics BLEU-4:", float(metric(preds, target_lists)))
except ImportError:
    print("Optional torchmetrics not installed; NLTK BLEU remains the main metric.")

# Optional CIDEr/ROUGE can be added with pycocoevalcap; install separately if needed.
try:
    import pycocoevalcap  # noqa: F401
    print("pycocoevalcap is installed; its COCO-style evaluator needs image-ID keyed dictionaries.")
except ImportError:
    print("Optional pycocoevalcap not installed; skipping CIDEr/ROUGE.")

## 17. Best and worst test captions by sentence BLEU

This ranks each generated caption against that image's five references. Sentence BLEU is useful for qualitative inspection but should not be treated as a stable per-image quality estimator.

In [ ]:
def rank_caption_examples(image_ids, hypotheses_by_id, clean_caption_map, idx2word=None):
    rows = []
    for image_id in image_ids:
        if image_id not in hypotheses_by_id:
            continue
        refs = references_as_tokens(image_id, clean_caption_map)
        hyp = hypotheses_by_id[image_id]
        score = sentence_bleu(refs, hyp, weights=(0.25, 0.25, 0.25, 0.25),
                              smoothing_function=SmoothingFunction().method1)
        rows.append({"image_id": image_id, "sentence_bleu4": score,
                     "generated": " ".join(hyp), "references": refs})
    return pd.DataFrame(rows).sort_values("sentence_bleu4", ascending=False).reset_index(drop=True)


def show_ranked_caption_rows(rows_df, n=5, worst=False):
    selected = rows_df.tail(n).sort_values("sentence_bleu4") if worst else rows_df.head(n)
    for _, row in selected.iterrows():
        print(f"{row['image_id']} | sentence BLEU-4={row['sentence_bleu4']:.4f}")
        print("Generated:", row["generated"])
        for j, ref in enumerate(row["references"], 1):
            print(f"  Ref {j}: {' '.join(ref)}")
        print()
        path = image_path_for(row["image_id"])
        if path.exists():
            plt.figure(figsize=(5, 3))
            plt.imshow(Image.open(path).convert("RGB"))
            plt.title(f"{row['image_id']} | BLEU-4 {row['sentence_bleu4']:.3f}")
            plt.axis("off")
            plt.show()

if test_greedy is not None:
    ranked_test = rank_caption_examples(test_ids, test_greedy, captions_clean)
    print("Five best captions:")
    show_ranked_caption_rows(ranked_test, n=5, worst=False)
    print("Five worst captions:")
    show_ranked_caption_rows(ranked_test, n=5, worst=True)
else:
    print("Generate test captions first to rank examples.")

In [ ]:
def show_random_test_examples(image_ids, hypotheses_by_id, clean_caption_map, n=5):
    candidates = [x for x in image_ids if x in hypotheses_by_id and image_path_for(x).exists()]
    if not candidates:
        print("No generated test captions available to display.")
        return
    for image_id in random.sample(candidates, k=min(n, len(candidates))):
        with Image.open(image_path_for(image_id)) as im:
            image = im.convert("RGB")
        plt.figure(figsize=(7, 4))
        plt.imshow(image)
        plt.axis("off")
        plt.title(f"Test image: {image_id}")
        plt.show()
        print("Generated:", " ".join(hypotheses_by_id[image_id]))
        for j, ref in enumerate(clean_caption_map[image_id], 1):
            print(f"Reference {j}:", ref)
        print()

if test_greedy is not None:
    show_random_test_examples(test_ids, test_greedy, captions_clean, n=5)
else:
    print("Run test generation first to show five random test-image captions.")

# Part VI — Attention visualization

## 18. Upsample and overlay the attention map

Each timestep's `alpha` has length L. For VGG16 with 196 locations, reshape to 14×14 and upsample to 224×224. The same helper works for a native 7×7 ResNet map by inferring the grid size. The upsampled map is smoothed with a Gaussian filter for visualization only; this does not change model attention.

In [ ]:
from scipy.ndimage import zoom, gaussian_filter


def upsample_attention(alpha, output_size=224, gaussian_sigma=8.0):
    alpha = np.asarray(alpha, dtype=np.float32).reshape(-1)
    grid_size = int(round(math.sqrt(alpha.size)))
    if grid_size * grid_size != alpha.size:
        raise ValueError(f"Attention length {alpha.size} is not a square grid")
    grid = alpha.reshape(grid_size, grid_size)
    scale = output_size / grid_size
    upsampled = zoom(grid, zoom=(scale, scale), order=1)
    # scipy zoom may differ by one pixel due to rounding; resize if needed.
    if upsampled.shape != (output_size, output_size):
        from PIL import Image as PILImage
        arr = (upsampled - upsampled.min()) / (np.ptp(upsampled) + 1e-8)
        upsampled = np.asarray(PILImage.fromarray((arr * 255).astype(np.uint8)).resize(
            (output_size, output_size), resample=PILImage.Resampling.BILINEAR
        ), dtype=np.float32) / 255.0
    if gaussian_sigma and gaussian_sigma > 0:
        upsampled = gaussian_filter(upsampled, sigma=gaussian_sigma)
    return upsampled


def plot_attention_grid(image, token_ids, attention_maps, vocab, max_cols=5):
    """Show an image and per-generated-word spatial attention overlays."""
    image = image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    words_and_maps = []
    for token_id, alpha in zip(token_ids[1:], attention_maps):      # skip BOS
        token_id = int(token_id)
        if token_id in (PAD_ID, BOS_ID, EOS_ID):
            continue
        words_and_maps.append((vocab.itos[token_id], alpha))
    if not words_and_maps:
        print("No ordinary generated words with attention maps.")
        return
    n = len(words_and_maps)
    cols = min(max_cols, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (word, alpha) in zip(axes.flat, words_and_maps):
        ax.imshow(image)
        heatmap = upsample_attention(alpha, output_size=IMAGE_SIZE, gaussian_sigma=8.0)
        ax.imshow(heatmap, alpha=0.6, cmap="gray")
        ax.set_title(word)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


ATTENTION_DEMO_IMAGE_ID = None  # e.g. test_ids[0]
if ATTENTION_DEMO_IMAGE_ID is not None and CHECKPOINT_PATH.exists():
    load_captioner_checkpoint(model, CHECKPOINT_PATH)
    sequence, maps = greedy_decode_features(model, test_store.get(ATTENTION_DEMO_IMAGE_ID), vocab)
    with Image.open(image_path_for(ATTENTION_DEMO_IMAGE_ID)) as im:
        plot_attention_grid(im, sequence, maps, vocab)
else:
    print("Set ATTENTION_DEMO_IMAGE_ID to a test filename to visualize per-word maps.")


if test_greedy is not None and CHECKPOINT_PATH.exists() and not ranked_test.empty:
    load_captioner_checkpoint(model, CHECKPOINT_PATH)
    best_id = ranked_test.iloc[0]["image_id"]
    worst_id = ranked_test.iloc[-1]["image_id"]
    for label, img_id in [("Higher", best_id), ("Lower", worst_id)]:
        print(f"{label} sentence-BLEU example:", img_id, "->", " ".join(test_greedy[img_id]))
        seq, maps = greedy_decode_features(model, test_store.get(img_id))
        with Image.open(image_path_for(img_id)) as im:
            plot_attention_grid(im, seq, maps, vocab)
else:
    print("Run test generation to compare high- and low-scoring attention maps.")

# Part VII — Research extensions and ablations

## 20. Ablation plan

Ablations should keep the train/validation/test split, vocabulary, stopping criterion, and decoding settings fixed. For each row, rebuild the decoder from a fresh initialisation; for encoder changes, extract and use a separate cache because the feature dimension/spatial grid changes.

| Ablation | Configuration | Expected implementation change |
|---|---|---|
| VGG16 vs ResNet50 | VGG: 196×512; native ResNet: 49×2048 | Rebuild encoder/cache and set `encoder_dim` from its metadata |
| Doubly-stochastic term | `lambda_ds=1.0` vs `0.0` | Same weights/setup; change loss coefficient |
| Beta gate | `use_beta=True` vs `False` | Rebuild decoder |
| No attention baseline | `use_attention=True` vs `False` | Mean-pool spatial features at every step; use `lambda_ds=0` |
| Embedding/hidden dimension | e.g. embed 256/512, hidden 512/768 | Rebuild decoder and optimiser |
| LSTM vs GRU | `cell_type="lstm"` vs `"gru"` | Rebuild decoder; GRU has no cell state |
| GloVe | initialise embedding from matching-dimensional GloVe | Rebuild with embedding dimension matching vectors |

The following config helper makes decoder variants reproducible. It does not automatically run all the costly experiments; use `fit_captioner(...)` and record the returned BLEU results for each variant.

In [ ]:
# Encoder-switch example (run separately from the primary cache):
# resnet_encoder = CNNFeatureExtractor("resnet50", pretrained=True, output_size=None).to(device)
# RESNET_FEATURE_PATH = FLICKR_ROOT / "flickr8k_resnet50_features.fp16.mmap"
# RESNET_META_PATH = FLICKR_ROOT / "flickr8k_resnet50_features.meta.json"
# resnet_meta = extract_feature_cache(
#     resnet_encoder, sorted(captions_with_tokens), IMAGE_DIR, image_transform,
#     RESNET_FEATURE_PATH, RESNET_META_PATH, batch_size=32,
# )
# resnet_feature_store = FeatureStore(RESNET_FEATURE_PATH, RESNET_META_PATH)
# Expected native ResNet50 feature shape: (N, 49, 2048). Build new datasets/loaders from this store,
# then train a fresh decoder with encoder_dim=2048. Compare BLEU on the same validation/test IDs.

In [ ]:
def make_captioner_variant(
    vocab_size, encoder_dim, embed_dim=256, hidden_dim=512, attention_dim=256,
    use_beta=True, use_attention=True, cell_type="lstm", dropout=0.5,
):
    return AttentionCaptioningDecoder(
        vocab_size=vocab_size,
        encoder_dim=encoder_dim,
        embed_dim=embed_dim,
        hidden_dim=hidden_dim,
        attention_dim=attention_dim,
        dropout=dropout,
        pad_idx=PAD_ID,
        use_beta=use_beta,
        use_attention=use_attention,
        cell_type=cell_type,
    ).to(DEVICE)

ABLATION_CONFIGS = {
    "full_attention": {"use_attention": True, "use_beta": True, "cell_type": "lstm", "lambda_ds": 1.0},
    "no_doubly_stochastic": {"use_attention": True, "use_beta": True, "cell_type": "lstm", "lambda_ds": 0.0},
    # "no_beta_gate": {"use_attention": True, "use_beta": False, "cell_type": "lstm", "lambda_ds": 1.0},
    # "no_attention_mean_pool": {"use_attention": False, "use_beta": False, "cell_type": "lstm", "lambda_ds": 0.0},
    # "gru_attention": {"use_attention": True, "use_beta": True, "cell_type": "gru", "lambda_ds": 1.0},
    # "embedding_512": {"use_attention": True, "use_beta": True, "cell_type": "lstm", "lambda_ds": 1.0, "embed_dim": 512, "hidden_dim": 512},
    # "hidden_768": {"use_attention": True, "use_beta": True, "cell_type": "lstm", "lambda_ds": 1.0, "embed_dim": 256, "hidden_dim": 768},
}

def run_decoder_ablation_sweep(
    configs=ABLATION_CONFIGS,
    epochs=MAX_EPOCHS,
    learning_rate=LEARNING_RATE,
    optimizer_name="RMSprop",
    patience=EARLY_STOPPING_PATIENCE,
):
    """Train separate decoder variants on the current feature cache and report validation BLEU."""
    encoder_dim = int(train_loader.dataset.data.shape[-1])
    rows = []
    for name, cfg in configs.items():
        print(f"\n=== Ablation: {name} ===")
        variant = make_captioner_variant(
            vocab_size=len(vocab),
            encoder_dim=encoder_dim,
            embed_dim=cfg.get("embed_dim", EMBED_DIM),
            hidden_dim=cfg.get("hidden_dim", HIDDEN_DIM),
            attention_dim=ATTENTION_DIM,
            use_beta=cfg.get("use_beta", True),
            use_attention=cfg.get("use_attention", True),
            cell_type=cfg.get("cell_type", "lstm"),
        )
        variant_checkpoint = CHECKPOINT_PATH.with_name(f"{CHECKPOINT_PATH.stem}_{name}.pt")
        hist, _ = fit_captioner(
            variant, train_loader, val_loader, vocab,
            epochs=epochs, learning_rate=learning_rate, optimizer_name=optimizer_name,
            lambda_ds=cfg.get("lambda_ds", 1.0), checkpoint_path=variant_checkpoint,
            patience=patience,
        )
        load_captioner_checkpoint(variant, variant_checkpoint)
        scores, _, _ = evaluate_corpus_bleu(variant, val_loader, vocab, beam_size=1)
        rows.append({"ablation": name, **scores, "epochs_run": len(hist)})
    return pd.DataFrame(rows).sort_values("BLEU-4", ascending=False).reset_index(drop=True)


RUN_ABLATION_SWEEP = True
if RUN_ABLATION_SWEEP:
    ablation_results = run_decoder_ablation_sweep(epochs=1)
    display(ablation_results)
else:
    print("Ablation sweep is disabled by default; set RUN_ABLATION_SWEEP=True to run it.")